# シンプルな投票確認サンプル
 投票した内容の確認のためのサンプル

## Import

In [ ]:
import requests
import pandas as pd
import joblib

import time
from datetime import datetime, timedelta

## Get Data
 当日配布データのAPIを利用してデータを取得.

In [ ]:
# 今日の日付：ベットする日付
target_date = "20250913"

In [ ]:
url = "https://172.192.40.114/data"

headers = {
    # 認証キー:当日データAPIを使う時は共通
    "api-key": "AI_Keiba_2025",
    # "Racecards" or "Odds"
    "type_of_data": "Racecards",
    # Racecardsのときは日付, OddsのときはYYYYMMDDJJRRを入力:JJは場所コード, RRはレース番号
    # 場所コード："札幌", "函館", "福島", "新潟", "東京", "中山", "中京", "京都", "阪神", "小倉"は順番に01, 02, ..., 10
    "id": target_date
}

# --- リクエスト送信 ---
# 形だけ証明書なのでverify=Falseにする: Warningは出るが無視でオッケー
response = requests.get(url, headers=headers, verify=False)
res_data = response.json()

/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '172.192.40.114'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
timetable = pd.DataFrame(res_data["data"]["timetable"])
runtable = pd.DataFrame(res_data["data"]["runtable"])

In [ ]:
timetable.head()

,year,month,day,place,race_num,start_time,class_code,track_code
0,2025,9,13,中山,1,09:45,7,24
1,2025,9,13,中山,2,10:15,7,18
2,2025,9,13,中山,3,10:45,7,24
3,2025,9,13,中山,4,11:20,131,54
4,2025,9,13,中山,5,12:10,15,18


In [ ]:
runtable.head()

,place,race_num,horse_num,dist,horse,sex,age,jockey,loaf_weight,father,mother,id,waku_num,race_id
0,中山,1,1,1800,ジャストビコーズ,牝,2,今村聖奈,53.0,インディチャンプ,イナズマアマリリス,23104065,1,202509130604030101
1,中山,1,2,1800,パレスノハレブタイ,牝,2,津村明秀,55.0,デクラレーションオブウォー,ルミノハレブタイ,23103747,2,202509130604030102
2,中山,1,3,1800,ラスエル,牝,2,内田博幸,55.0,クリソベリル,ラスエモーショネス,23107446,3,202509130604030103
3,中山,1,4,1800,スピネルテソーロ,牝,2,佐々木大,55.0,スワーヴリチャード,トロントテソーロ,23106745,4,202509130604030104
4,中山,1,5,1800,リワードシュテルン,牝,2,舟山瑠泉,52.0,モーニン,リワードアメイン,23102267,5,202509130604030105


## Creating race id for vote

In [ ]:
runtable["race_id_vote"] = runtable["race_id"].apply(lambda x: int(str(x)[0:4] + str(x)[8:16]))

## Check Vote

In [ ]:
##### netkeibaのアカウント情報
# 皆さんのアカウント情報に書き換えて下さい
login_id = ""
password = ""

In [ ]:
def vote_api_login_fun(login_id, password):
    url = "https://masters.netkeiba.com/ai2025_student/api/login"
    headers = {
        "Content-Type": "application/json"
    }
    payload = {
        "login_id": login_id,
        "password": password
    }

    res = requests.post(url, headers=headers, json=payload)
    if res.status_code == 200:
        print("ログイン成功")
        return res.json()["data"]["access_token"]
    else:
        print(f"ログイン失敗: {res.status_code}")
        return {}

def vote_api_logout_fun(access_token):
    url = "https://masters.netkeiba.com/ai2025_student/api/logout"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    res = requests.post(url, headers=headers)
    if res.status_code == 200:
        print("ログアウト成功")
    else:
        print(f"ログアウト失敗: {response.status_code}")

def vote_api_bet_fun(bet_data_json, access_token):
    url = "https://masters.netkeiba.com/ai2025_student/api/bet"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}"
    }
    payload = {
        "bet_data": [bet_data_json]
    }
    # POSTリクエスト送信
    res = requests.post(url, headers=headers, json=payload)

    # レスポンス確認
    if res.status_code == 200:
        print("ベット成功")
        return res.json()
    else:
        print(f"エラー: {res.status_code}")
        print(res.text)
        return {}

def get_odds_rt_fun(race_id):
    url = "https://172.192.40.114/data"
    headers = {
        # 認証キー:当日データAPIを使う時は共通
        "api-key": "AI_Keiba_2025",
        # "Racecards" or "Odds"
        "type_of_data": "Odds",
        # Racecardsのときは日付, OddsのときはYYYYMMDDJJRRを入力:JJは場所コード, RRはレース番号
        # 場所コード："札幌", "函館", "福島", "新潟", "東京", "中山", "中京", "京都", "阪神", "小倉"は順番に01, 02, ..., 10
        "id": race_id
    }

    # 形だけ証明書なのでverify=Falseにする: Warningは出るが無視でオッケー
    res = requests.get(url, headers=headers, verify=False)

    # --- 結果表示 ---
    if res.status_code == 200:
        data = res.json()
        print("成功:", data["message"])
        return data
    else:
        print(f"エラー: {res.status_code}")
        print(res.text)
        return {}

def vote_api_check_fun(race_id, access_token):

    url = "https://masters.netkeiba.com/ai2025_student/api/bet"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}"
    }
    payload = {"race_id[]": race_id}
    # リクエスト送信
    res = requests.get(url, headers=headers, params=payload)

    return res

### レースを選択

In [ ]:
target_place = "中山"
target_race_num = 6 # 数値
target_race = timetable[(timetable["place"]==target_place) & (timetable["race_num"]==target_race_num)]
target_race

,year,month,day,place,race_num,start_time,class_code,track_code
5,2025,9,13,中山,6,12:40,15,17


### 投票した内容確認
- 投票をしていないrace_idを指定すると"status": "NG"が返ってくる

In [ ]:
# 午後のレースにbetするサンプル
rows = target_race.copy()

# 投票するレースのid
race_id_vote = str((runtable[(runtable["place"]==target_place) & (runtable["race_num"]==target_race_num)]["race_id_vote"].values[0]))

###### 投票確認
# ログイン
access_token = vote_api_login_fun(login_id, password)

# ベット
res = vote_api_check_fun(race_id_vote, access_token)
print(res.text)

# ログアウト
vote_api_logout_fun(access_token)

ログイン成功
{"status":"OK","data":[{"race_id":"202506040306","mark":{"1":1},"bet":[{"bet_id":"b1_c0_1","money":100}]}]}
ログアウト成功
